# 01 — Exploration du corpus — ce qu'une règle de surface expliquerait déjà

**Projet** : Routage de tickets de support par TF-IDF — la référence explicable

## Objectifs pédagogiques

1. Décrire un corpus multiclasse : effectifs par classe, par style, par canal, longueurs, dates.
1. Mesurer le **niveau à battre** : classe majoritaire, tirage stratifié, et la règle à mots-clés publiée avec le corpus.
1. Chiffrer les **distracteurs déclarés** (priorité et canal) au lieu d'affirmer qu'ils sont inutiles.
1. Vérifier la propriété qui rend le score crédible : les formulations du test ne sont jamais vues à l'entraînement.

## 1. Mise en place

Le notebook charge **la configuration du projet**, pas une copie : la même que `python -m src.main`. Les valeurs affichées plus bas sont donc celles du run réel, à la taille du corpus près (les notebooks travaillent sur un corpus réduit quand `data/raw` est vide, pour rester exécutables en quelques secondes).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (320 tickets) quand `data/raw` est
# vide : l'exécution reste rapide. Si le corpus de référence a été généré par `make data`, il est
# utilisé tel quel — les valeurs absolues d'un notebook ne sont pas celles de la référence.
NB_DOCUMENTS = 320

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks lisent le corpus du projet et écrivent leurs artefacts dans `outputs/notebooks` :
# ils ne doivent jamais écraser ceux produits par `make train` / `make evaluate`.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PATHS.data_dir,
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
TARGET_COLUMN = str(CONFIG.data.target or "label")
LABELS = [
    "facturation",
    "livraison",
    "produit_defectueux",
    "remboursement",
    "compte_client",
    "autre",
]
STYLES = ["canonique", "paraphrase", "bruite"]

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.7)")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Classes           : {len(LABELS)} — {', '.join(LABELS)}")
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticTicketGenerator
from src.data.loaders import TextLabelLoader

LOADER = TextLabelLoader(
    PATHS,
    formats=CONFIG.data.formats,
    validation_enabled=CONFIG.data.validation.raw,
    lazy_validation=CONFIG.data.validation.lazy,
)
if LOADER.documents_path.exists():
    DOCUMENTS = LOADER.load_documents()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis data/raw : {len(DOCUMENTS)} tickets, graine {METADATA.get('seed')}")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté pour que les pipelines (train, evaluate) trouvent les mêmes fichiers que le
    # notebook ; `make data` régénère le corpus de référence.
    bundle = SyntheticTicketGenerator.from_config(CONFIG.data, seed=CONFIG.seed).generate()
    DOCUMENTS, METADATA = bundle.documents, bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_metadata(METADATA)
    print(f"data/raw absent : corpus de {len(DOCUMENTS)} tickets généré et persisté")

TRAIN = LOADER.split("train", frame=DOCUMENTS)
VALIDATION = LOADER.split("val", frame=DOCUMENTS)
CALIBRATION = LOADER.split("calibration", frame=DOCUMENTS)
TEST = LOADER.split("test", frame=DOCUMENTS)
# La calibration sert à **régler** (seuil d'automatisation), la validation à **choisir**
# (algorithme, rééquilibrage), et le test à **juger** — une seule fois, à la fin.
print(
    "splits :",
    {
        "train": len(TRAIN),
        "val": len(VALIDATION),
        "calibration": len(CALIBRATION),
        "test": len(TEST),
    },
)

LABEL_COUNTS = DOCUMENTS[TARGET_COLUMN].value_counts()
MAJORITY_LABEL = str(LABEL_COUNTS.index[0])
MINORITY_LABEL = str(LABEL_COUNTS.index[-1])
print(
    f"classe majoritaire : {MAJORITY_LABEL} ({LABEL_COUNTS.iloc[0] / len(DOCUMENTS):.1%}) "
    f"| classe minoritaire : {MINORITY_LABEL} ({LABEL_COUNTS.iloc[-1] / len(DOCUMENTS):.1%})"
)

**Ce que la cellule précédente vient d'établir**

- Le corpus, ses métadonnées et ses splits viennent du **même générateur** que celui de `make data` : le notebook ne fabrique pas un jeu d'exemple à part.
- Les quatre splits sont **écrits dans le corpus** : deux exécutions voient les mêmes lignes de test, donc les chiffres publiés sont comparables.
- Le corpus réduit des notebooks est persisté dans `data/raw`, comme celui de `make data` : les pipelines lancés ensuite lisent exactement ce que le notebook a lu.

## 2. Effectifs : les classes, les styles, les canaux

Trois tableaux, trois questions différentes. Les **classes** sont la cible du modèle. Les **styles** disent *comment* le ticket est écrit : c'est le découpage qui distinguera plus tard « lire un mot » de « comprendre une demande ». Les **canaux** et la **priorité** sont des distracteurs déclarés : ils sont tirés indépendamment du libellé, et la section 5 le chiffre.

In [ ]:
summary = pd.DataFrame(
    {
        "tickets": LABEL_COUNTS,
        "part": (LABEL_COUNTS / len(DOCUMENTS)).round(4),
        "split train": TRAIN[TARGET_COLUMN].value_counts(),
        "split test": TEST[TARGET_COLUMN].value_counts(),
    }
).rename_axis("classe")
display(summary)

shares = pd.DataFrame(
    {
        "style": (DOCUMENTS["style"].value_counts() / len(DOCUMENTS)).round(4),
        "source": (DOCUMENTS["source"].value_counts() / len(DOCUMENTS)).round(4),
        "priority": (DOCUMENTS["priority"].value_counts() / len(DOCUMENTS)).round(4),
    }
)
display(shares)

**Ce qu'il faut retenir**

- Aucune classe n'écrase les autres, mais la plus fréquente pèse plusieurs fois la plus rare : une exactitude globale récompenserait un modèle qui ignorerait la classe difficile — d'où la F1 **macro** comme métrique principale.
- Les trois styles et les quatre canaux sont dans les proportions annoncées par le manifeste : le corpus réduit respecte les mêmes parts que le corpus de référence.
- Les styles sont répartis dans les deux splits : le test contient donc bien des tickets canoniques, des paraphrases et des tickets bruités.

## 3. Croisement des styles et des classes et longueurs

Un segment n'est lisible que s'il est rempli. On vérifie donc que chaque style couvre chaque classe — et on regarde la longueur des tickets, parce qu'un classifier lexical peut apprendre « les tickets longs sont des remboursements » sans rien comprendre à la demande.

In [ ]:
cross = pd.crosstab(DOCUMENTS["style"], DOCUMENTS[TARGET_COLUMN], margins=True)
display(cross)

lengths = (
    DOCUMENTS.groupby([TARGET_COLUMN, "style"], observed=True)["n_tokens"]
    .agg(["size", "mean", "median", "max"])
    .round(2)
)
display(lengths.head(12))
print(
    "longueur moyenne :",
    round(float(DOCUMENTS["n_tokens"].mean()), 2),
    "| médiane :",
    float(DOCUMENTS["n_tokens"].median()),
)
print("part des tickets sous 20 tokens :", f"{float((DOCUMENTS['n_tokens'] < 20).mean()):.1%}")

In [ ]:
from src.visualization.plots import plot_lengths

FIGURE = plot_lengths(DOCUMENTS, NB_PATHS.figures_dir / "01_lengths.png")
print(f"figure écrite : {FIGURE.relative_to(PROJECT_ROOT)}")

**Ce qu'il faut retenir**

- Chaque couple (classe, style) existe, donc chaque segment de la future ventilation aura de quoi mesurer quelque chose.
- Les longueurs se chevauchent d'une classe à l'autre : la longueur seule ne peut pas trancher — c'est ce que le test de raccourci vérifie numériquement plus bas.
- Les tickets bruités sont mécaniquement plus longs (politesse et signature) : leur exactitude devra être lue à côté de celle des tickets canoniques, jamais seule.

## 4. Le niveau à battre

Un score n'a de sens que situé. On mesure ici les trois références du projet : la classe majoritaire (le plancher absolu), le tirage stratifié (ce que le hasard bien fait obtient) et la **règle à mots-clés** publiée avec le corpus — celle qu'un gestionnaire de support écrirait en dix lignes dans un tableur.

In [ ]:
from src.data.generators import class_keywords, keyword_baseline_accuracy
from src.training.metrics import majority_baseline, stratified_baseline

truth = DOCUMENTS[TARGET_COLUMN].astype(str)
baselines = {
    "classe majoritaire": majority_baseline(truth),
    "tirage stratifie": stratified_baseline(truth, seed=CONFIG.seed),
}
baseline_table = pd.DataFrame(baselines).T
display(baseline_table.round(4))

rule_accuracy = keyword_baseline_accuracy(DOCUMENTS)
keywords = class_keywords()
rule_table = pd.DataFrame(
    {
        "classe": list(keywords),
        "mots-clés": [
            ", ".join(terms) if terms else "(aucun : la classe « autre »)"
            for terms in keywords.values()
        ],
    }
).set_index("classe")
display(rule_table)
print(
    f"règle à mots-clés : exactitude {rule_accuracy:.4f} "
    f"(classe majoritaire : {baselines['classe majoritaire']['accuracy']:.4f})"
)

**Ce qu'il faut retenir**

- Le plancher est bas : la classe majoritaire vaut une fraction des tickets, et la F1 macro de ce plancher est catastrophique — c'est la métrique qui protège les classes rares.
- La règle à mots-clés **bat largement** la classe majoritaire : un projet qui ne la dépasse pas n'aurait rien apporté à l'équipe support, et c'est écrit dans le rapport.
- La classe « autre » n'a aucun mot-clé par construction : aucune règle de surface ne peut la prédire, et c'est là que la F1 macro se joue.

## 5. Distracteurs déclarés : le test de raccourci

La priorité déclarée par le client et le canal d'arrivée sont tentants : ils sont disponibles avant la lecture du texte. S'ils suffisaient à prédire le libellé, le projet serait un exercice de tableur. On mesure donc le gain de la meilleure règle à un seuil construite sur chacun d'eux, et on le compare au niveau de la classe majoritaire — pas à zéro.

In [ ]:
from src.features.build_features import TEXT_FEATURES, shortcut_scores

candidates = [*TEXT_FEATURES, "source", "priority"]
scores = shortcut_scores(DOCUMENTS)
shortcut_table = (
    pd.DataFrame({"gain": [scores[name] for name in candidates]}, index=candidates)
    .assign(marge_vs_plancher=lambda frame: (frame["gain"] - scores["majority_baseline"]).round(4))
    .sort_values("gain", ascending=False)
    .round(4)
)
display(shortcut_table)

print(f"niveau de la classe majoritaire : {scores['majority_baseline']:.4f}")
print(f"meilleur gain de surface        : {scores['shortcut_margin']:.4f}")
assert scores["source"] < scores["majority_baseline"] + 0.05
assert scores["priority"] < scores["majority_baseline"] + 0.05
print("contrat vérifié : ni le canal ni la priorité ne prédisent le libellé")

**Ce qu'il faut retenir**

- Le canal et la priorité ne font pas mieux qu'une règle constante : ce sont bien des distracteurs, et le test de raccourci le vérifie à chaque exécution du notebook.
- Les traits de forme (longueur, politesse, chiffres) restent sous le niveau du plancher + 0,05 : mémoriser la forme des tickets ne suffit pas à les router.
- Ce tableau est le même que celui que le rapport d'évaluation publie : le notebook et la CI regardent le même instrument.

## 6. La formulation du test n'est jamais vue à l'entraînement

Dernière vérification, et la plus importante : chaque pool de formulations (classe et style) est coupé en deux, quelques tournures étant réservées aux splits d'évaluation. Sans cette réserve, un modèle lexical se contenterait de recopier des gabarits mémorisés et la F1 publiée serait un score de recopie.

In [ ]:
import re

from src.data.generators import PRODUCTS
from src.preprocessing.transformers import normalise_text

ORDER = re.compile(r"cmd-\d{5}")


def wording(text: str) -> str:
    """Rend la formulation d'un ticket : ses valeurs tirées deviennent des jetons neutres."""
    lowered = normalise_text(str(text)).lower()
    for product in PRODUCTS:
        lowered = lowered.replace(product.lower(), "{produit}")
    return ORDER.sub("{commande}", lowered).strip()


rows = []
for (label, style), group in DOCUMENTS.groupby([TARGET_COLUMN, "style"], observed=True):
    learned = {wording(text) for text in group.loc[group["split"] == "train", "text"]}
    evaluated = {wording(text) for text in group.loc[group["split"] != "train", "text"]}
    rows.append(
        {
            "classe": label,
            "style": style,
            "formulations vues (train)": len(learned),
            "formulations évaluées": len(evaluated),
            "intersection": len(learned & evaluated),
        }
    )
holdout = pd.DataFrame(rows)
display(holdout.head(9))
assert (holdout["intersection"] == 0).all(), "une formulation du test a été vue à l'entraînement"
print(
    f"réserve déclarée par le manifeste : {METADATA.get('phrasing_holdout')} formulations "
    "par couple (classe, style)"
)
print("contrat vérifié : aucune formulation d'évaluation n'apparaît dans le train")

In [ ]:
example = TEST.loc[TEST["style"] == "paraphrase", "text"].iloc[0]
print("exemple de ticket du test (paraphrase) :")
print(" ", example)
print()
print("cinq tickets canoniques du train, pour comparer les tournures :")
for text in TRAIN.loc[TRAIN["style"] == "canonique", "text"].head(5):
    print(" -", text)

**Ce qu'il faut retenir**

- Les formulations du test sont disjointes de celles du train : un score élevé ne peut pas venir d'un gabarit mémorisé.
- La réserve est un paramètre du corpus (`data.corpus.phrasing_holdout`), pas une constante cachée dans le générateur.
- C'est cette propriété qui rend la comparaison avec la variante `transformers` honnête : les deux modèles sont jugés sur les mêmes tournures inédites.